# Notebook 02 — RibFrac CT Preprocessing

This notebook preprocesses RibFrac CT images and corresponding annotation-label volumes.

**Methodological principles**

- Patient-level splitting is performed before patch extraction.
- CT and label volumes are spatially validated before preprocessing.
- CT intensities are clipped and normalized.
- CT volumes use linear interpolation during resampling.
- Label volumes use nearest-neighbor interpolation to preserve annotation IDs.
- The test patient `RibFrac128` is used first to validate the complete preprocessing and saving pipeline.
- The complete training-set preprocessing loop is intentionally kept separate and is not executed automatically.


In [ ]:
# Cell 1 — Import libraries

import os
import gc
import json
import numpy as np
import pandas as pd
import nibabel as nib

from pathlib import Path
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split

from ribfrac_data_zenodo import (
    download_ribfrac_image,
    get_label_path,
    load_patient,
    validate_patient
)

print("RibFrac data module imported successfully.")
print("Libraries imported successfully.")


In [ ]:
# Cell 2 — Define directories

BASE_DIR = Path("/kaggle/working")

IMAGE_DIR = BASE_DIR / "ribfrac_images"
LABEL_DIR = BASE_DIR / "ribfrac_labels"

METADATA_DIR = BASE_DIR / "ribfrac_metadata"
INFO_PATH = METADATA_DIR / "ribfrac-train-info-1.csv"

PREPROCESSED_DIR = BASE_DIR / "preprocessed"
PREPROCESSED_IMAGE_DIR = PREPROCESSED_DIR / "images"
PREPROCESSED_LABEL_DIR = PREPROCESSED_DIR / "labels"

PREPROCESSED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_LABEL_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Directories:")
print("Images:", IMAGE_DIR)
print("Labels:", LABEL_DIR)
print("Metadata:", INFO_PATH)
print("Preprocessed images:", PREPROCESSED_IMAGE_DIR)
print("Preprocessed labels:", PREPROCESSED_LABEL_DIR)


In [ ]:
# Cell 3 — Check directories

print("Image directory exists:", IMAGE_DIR.exists())
print("Label directory exists:", LABEL_DIR.exists())
print("Metadata file exists:", INFO_PATH.exists())

print("\nPreprocessed directories:")
print("Images:", PREPROCESSED_IMAGE_DIR)
print("Labels:", PREPROCESSED_LABEL_DIR)


In [ ]:
# Cell 4 — Load RibFrac metadata

INFO_URL = "https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv"

if not INFO_PATH.exists():
    print("Metadata file not found in /kaggle/working.")
    print("Downloading RibFrac metadata...")

    import requests

    response = requests.get(INFO_URL, timeout=120)
    response.raise_for_status()

    INFO_PATH.write_bytes(response.content)

    print("Metadata downloaded successfully.")
else:
    print("Metadata file already exists.")

print("Metadata path:", INFO_PATH)
print("File size:", INFO_PATH.stat().st_size, "bytes")

info_df = pd.read_csv(INFO_PATH)

print("\nMetadata loaded successfully.")
print("Shape:", info_df.shape)
print("Columns:", info_df.columns.tolist())
display(info_df.head())


In [ ]:
# Cell 5 — Create patient-level split

# RibFrac Part 1 contains 300 unique labelled patients.
patient_ids = sorted(info_df["public_id"].unique())

print("Total unique patients:", len(patient_ids))

# 80% development / 20% test
train_patients, test_patients = train_test_split(
    patient_ids,
    test_size=0.20,
    random_state=42
)

# 80% training / 20% validation from the development set
train_patients, val_patients = train_test_split(
    train_patients,
    test_size=0.20,
    random_state=42
)

print("\nPatient-level split:")
print("Train:", len(train_patients))
print("Validation:", len(val_patients))
print("Test:", len(test_patients))


In [ ]:
# Cell 6 — Verify patient-level split

train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

print("Train ∩ Validation:", train_set & val_set)
print("Train ∩ Test:", train_set & test_set)
print("Validation ∩ Test:", val_set & test_set)

assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

print("\nPatient-level leakage check: PASS")


In [ ]:
# Cell 7 — Save patient-level split

SPLIT_PATH = BASE_DIR / "patient_split.csv"

split_df = pd.DataFrame({
    "public_id": (
        train_patients +
        val_patients +
        test_patients
    ),
    "split": (
        ["train"] * len(train_patients) +
        ["validation"] * len(val_patients) +
        ["test"] * len(test_patients)
    )
})

split_df.to_csv(SPLIT_PATH, index=False)

print("Patient split saved:")
print(SPLIT_PATH)

print("\nSplit distribution:")
print(split_df["split"].value_counts())


In [ ]:
# Cell 8 — Define preprocessing parameters

HU_MIN = -1000.0
HU_MAX = 400.0

TARGET_SPACING = (1.0, 1.0, 1.0)

print("HU range:", HU_MIN, "to", HU_MAX)
print("Target spacing:", TARGET_SPACING)


In [ ]:
# Cell 9 — Load a patient

def load_patient_for_preprocessing(patient_id):
    """
    Load one patient's CT and corresponding annotation-label volume.
    """

    ct_path = download_ribfrac_image(patient_id)
    label_path = get_label_path(patient_id)

    ct_img = nib.load(str(ct_path))
    label_img = nib.load(str(label_path))

    ct_volume = ct_img.get_fdata(dtype=np.float32)
    label_volume = label_img.get_fdata(dtype=np.float32)

    return {
        "patient_id": patient_id,
        "ct_path": ct_path,
        "label_path": label_path,
        "ct_img": ct_img,
        "label_img": label_img,
        "ct_volume": ct_volume,
        "label_volume": label_volume
    }

print("Patient loading function defined.")


In [ ]:
# Cell 10 — Spatial validation

def validate_spatial_alignment(ct_img, label_img):
    """
    Validate shape, voxel spacing, and affine alignment.
    """

    ct_shape = ct_img.shape
    label_shape = label_img.shape

    ct_spacing = ct_img.header.get_zooms()[:3]
    label_spacing = label_img.header.get_zooms()[:3]

    shape_match = ct_shape == label_shape
    spacing_match = np.allclose(
        ct_spacing,
        label_spacing,
        atol=1e-5
    )
    affine_match = np.allclose(
        ct_img.affine,
        label_img.affine,
        atol=1e-4
    )

    return {
        "shape_match": shape_match,
        "spacing_match": spacing_match,
        "affine_match": affine_match,
        "ct_shape": ct_shape,
        "label_shape": label_shape,
        "ct_spacing": ct_spacing,
        "label_spacing": label_spacing
    }

print("Spatial validation function defined.")


In [ ]:
# Cell 11 — CT intensity preprocessing

def preprocess_ct_intensity(
    ct_volume,
    hu_min=HU_MIN,
    hu_max=HU_MAX
):
    """
    Clip CT intensities and normalize them to [0, 1].
    """

    ct_volume = np.asarray(
        ct_volume,
        dtype=np.float32
    )

    ct_volume = np.clip(
        ct_volume,
        hu_min,
        hu_max
    )

    ct_volume = (
        (ct_volume - hu_min)
        / (hu_max - hu_min)
    )

    return ct_volume.astype(np.float32)

print("CT intensity preprocessing function defined.")


In [ ]:
# Cell 12 — Calculate resampling factors

def calculate_zoom_factors(
    original_spacing,
    target_spacing
):
    """
    Calculate zoom factors for spatial resampling.

    factor = original spacing / target spacing
    """

    original_spacing = np.asarray(
        original_spacing,
        dtype=np.float32
    )

    target_spacing = np.asarray(
        target_spacing,
        dtype=np.float32
    )

    return original_spacing / target_spacing

print("Resampling-factor function defined.")


In [ ]:
# Cell 13 — Resample CT

def resample_ct(
    ct_volume,
    original_spacing,
    target_spacing=TARGET_SPACING
):
    """
    Resample CT using linear interpolation.
    """

    zoom_factors = calculate_zoom_factors(
        original_spacing,
        target_spacing
    )

    resampled_ct = zoom(
        ct_volume,
        zoom=zoom_factors,
        order=1
    )

    return resampled_ct.astype(np.float32)

print("CT resampling function defined.")


In [ ]:
# Cell 14 — Resample label

def resample_label(
    label_volume,
    original_spacing,
    target_spacing=TARGET_SPACING
):
    """
    Resample annotation labels using nearest-neighbor
    interpolation so that label IDs are preserved.
    """

    zoom_factors = calculate_zoom_factors(
        original_spacing,
        target_spacing
    )

    resampled_label = zoom(
        label_volume,
        zoom=zoom_factors,
        order=0
    )

    return resampled_label.astype(np.int16)

print("Label resampling function defined.")


In [ ]:
# Cell 15 — Complete preprocessing function

def preprocess_patient(patient_id):

    print("=" * 60)
    print(f"Processing patient: {patient_id}")
    print("=" * 60)

    # 1. Load patient
    patient = load_patient_for_preprocessing(patient_id)

    ct_img = patient["ct_img"]
    label_img = patient["label_img"]

    ct_volume = patient["ct_volume"]
    label_volume = patient["label_volume"]

    # 2. Validate spatial alignment
    validation = validate_spatial_alignment(
        ct_img,
        label_img
    )

    if not validation["shape_match"]:
        raise ValueError(
            f"CT/label shape mismatch for {patient_id}"
        )

    if not validation["spacing_match"]:
        raise ValueError(
            f"CT/label spacing mismatch for {patient_id}"
        )

    if not validation["affine_match"]:
        raise ValueError(
            f"CT/label affine mismatch for {patient_id}"
        )

    print("Spatial validation: PASS")

    # 3. Original spacing
    original_spacing = ct_img.header.get_zooms()[:3]

    print("Original spacing:", original_spacing)

    # 4. CT intensity preprocessing
    ct_volume = preprocess_ct_intensity(ct_volume)

    print(
        "CT intensity range after normalization:",
        float(ct_volume.min()),
        "to",
        float(ct_volume.max())
    )

    # 5. Resample CT
    ct_resampled = resample_ct(
        ct_volume,
        original_spacing,
        TARGET_SPACING
    )

    # 6. Resample label
    label_resampled = resample_label(
        label_volume,
        original_spacing,
        TARGET_SPACING
    )

    # 7. Output validation
    print("Resampled CT shape:", ct_resampled.shape)
    print("Resampled label shape:", label_resampled.shape)

    print(
        "CT range:",
        float(ct_resampled.min()),
        "to",
        float(ct_resampled.max())
    )

    print(
        "Label values:",
        np.unique(label_resampled)
    )

    assert ct_resampled.shape == label_resampled.shape

    return {
        "patient_id": patient_id,
        "ct": ct_resampled,
        "label": label_resampled,
        "original_spacing": original_spacing,
        "target_spacing": TARGET_SPACING
    }

print("Complete preprocessing function defined.")


In [ ]:
# Cell 16 — Test preprocessing with RibFrac128

test_patient = preprocess_patient("RibFrac128")


In [ ]:
# Cell 17 — Validate label integrity

unique_labels = np.unique(
    test_patient["label"]
)

print("Unique label values:")
print(unique_labels)

if np.all(
    unique_labels == unique_labels.astype(int)
):
    print("Label integrity check: PASS")
else:
    print("Label integrity check: FAIL")

assert np.all(
    unique_labels == unique_labels.astype(int)
)


In [ ]:
# Cell 18 — Check CT and label dimensions

ct_shape = test_patient["ct"].shape
label_shape = test_patient["label"].shape

print("CT shape:", ct_shape)
print("Label shape:", label_shape)

assert ct_shape == label_shape

print("CT/label shape check: PASS")


In [ ]:
# Cell 19 — Save preprocessed patient

def save_preprocessed_patient(patient_data):

    patient_id = patient_data["patient_id"]

    ct = patient_data["ct"]
    label = patient_data["label"]

    ct_path = (
        PREPROCESSED_IMAGE_DIR
        / f"{patient_id}-image.npy"
    )

    label_path = (
        PREPROCESSED_LABEL_DIR
        / f"{patient_id}-label.npy"
    )

    np.save(ct_path, ct)
    np.save(label_path, label)

    print(f"Saved CT:    {ct_path}")
    print(f"Saved label: {label_path}")

    return ct_path, label_path

print("Save function defined.")


In [ ]:
# Cell 20 — Save RibFrac128

ct_path, label_path = save_preprocessed_patient(
    test_patient
)

print("\nSaved CT:", ct_path)
print("Saved label:", label_path)


In [ ]:
# Cell 21 — Verify saved files

saved_ct = np.load(ct_path)
saved_label = np.load(label_path)

print("Saved CT shape:", saved_ct.shape)
print("Saved label shape:", saved_label.shape)

print("Saved CT dtype:", saved_ct.dtype)
print("Saved label dtype:", saved_label.dtype)

print(
    "Saved CT range:",
    float(saved_ct.min()),
    "to",
    float(saved_ct.max())
)

print(
    "Saved label values:",
    np.unique(saved_label)
)

assert saved_ct.shape == test_patient["ct"].shape
assert saved_label.shape == test_patient["label"].shape

print("\nSaved-file verification: PASS")


In [ ]:
# Cell 22 — Check memory

print(
    "CT memory:",
    saved_ct.nbytes / (1024 ** 2),
    "MB"
)

print(
    "Label memory:",
    saved_label.nbytes / (1024 ** 2),
    "MB"
)


In [ ]:
# Cell 23 — Release RibFrac128 test arrays

del test_patient
del saved_ct
del saved_label

gc.collect()

print("Temporary arrays released from memory.")


## Training-set preprocessing

The following cell is intentionally **not executed automatically**.

The patient-level split has already been created above, and `train_patients` contains the 192 development-training patients from RibFrac Part 1.

Before processing all patients, verify that the single-patient pipeline above is correct. Full preprocessing will download and resample each CT and can require substantial disk space and time.


In [ ]:
# Cell 24 — Optional: preprocess the training set

PROCESS_TRAINING_SET = False

if PROCESS_TRAINING_SET:

    for i, patient_id in enumerate(
        train_patients,
        start=1
    ):

        print(
            f"\nPatient {i}/{len(train_patients)}"
        )
        print(f"Patient ID: {patient_id}")

        processed = preprocess_patient(
            patient_id
        )

        save_preprocessed_patient(
            processed
        )

        del processed
        gc.collect()

    print("\nTraining-set preprocessing completed.")

else:
    print(
        "Training-set preprocessing is disabled. "
        "Set PROCESS_TRAINING_SET = True only after "
        "the single-patient pipeline has been verified."
    )
